In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Padding, Stride, and Dilation

The convolutional layer of that section evaluates its kernel only
where the kernel fits inside the input, so its output is smaller than its
input by one less than the kernel size along each axis. For a single layer
this hardly matters. In a deep network the size of every output becomes a
design decision, and two needs pull in opposite directions. How do we keep a
stack of layers from cutting away the border of the image? And how do we
shrink the image on purpose, so that a unit near the top can see all of it
without a network of hundreds of layers?

A stack of ten layers with $5\times 5$ kernels shows the first problem. Each
layer trims two pixels from every side of its input, so a $240\times 240$
image comes out as $200\times 200$. The stack has removed 30% of the image's
area, all of it at the border. The second problem is the reverse. A network
that never shrinks its input computes a full-resolution output in every
layer, and each small kernel lets a unit see only a few pixels farther than
the layer below, so a unit needs very many layers to see the whole image.
A third need is for one kernel to see a wide region without the weights and
the computation of a large kernel.

Three settings of a convolutional layer address these needs. *Padding* adds a
border of extra pixels around the input, so that the kernel can be placed at
the edge. *Stride* moves the window by more than one pixel at a time.
*Dilation* spaces the kernel's weights apart. Each setting changes the output
shape by a formula we derive below, and each changes how far a unit in a deep
network sees. We stay with single-channel inputs, as in
that section, and add channels in that section.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Padding

### Boundary Pixels

Pixels near the border are underused even before a layer cuts them away.
the figure counts the windows that contain each pixel. With a
$k\times k$ kernel, a pixel at least $k-1$ pixels away from every border lies
in $k^2$ windows, a pixel on an edge in at most $k$, and a corner pixel in a
single one. A $5\times 5$ kernel covers each interior pixel in 25 windows but
a corner pixel in only one, so an object near the corner of an image barely
reaches the next layer. Each layer thus draws most of its evidence from the
center of its input, and the next layer receives an input from which the border has been
cut away.

![Number of windows that contain each pixel when a $1\times 1$, $2\times 2$, or $3\times 3$ kernel slides over a large input without padding. Interior pixels are covered $k^2$ times, corner pixels once.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-reuse.svg)

### Zero Padding

The remedy is to enlarge the input by a border of extra pixels before applying
the kernel, usually pixels of value zero. This is called *padding*. In
the figure, one row of zeros above and below and one column on
either side turn the $3\times 3$ input into a $5\times 5$ one, and the
$2\times 2$ kernel now produces a $4\times 4$ output instead of a
$2\times 2$ one. The shaded first output element is computed entirely from
padding and the input's corner: $0\times0+0\times1+0\times2+0\times3=0$.

![Two-dimensional cross-correlation with one pixel of zero padding on every side.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-pad.svg)

If we add $p_\textrm{h}$ rows of padding in total, split between top and
bottom, and $p_\textrm{w}$ columns in total, split between left and right,
the padded input has shape $(n_\textrm{h}+p_\textrm{h})\times
(n_\textrm{w}+p_\textrm{w})$, and the output has shape

$$(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+1)\times(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+1).$$

Each row of padding adds one row to the output, and each column adds one
column. Setting $p_\textrm{h}=k_\textrm{h}-1$ and $p_\textrm{w}=k_\textrm{w}-1$
gives the output the shape of the input. A stack of such layers keeps its
resolution however deep it is, which makes the shape of every layer easy to
predict.

The kernel size decides whether this padding can be split evenly. For an odd
$k_\textrm{h}$, the total $k_\textrm{h}-1$ is even, and we pad
$(k_\textrm{h}-1)/2$ rows on each side. The window that computes the output
`Y[i, j]` is then centered on the input `X[i, j]`: the output is aligned with
the input. For an even $k_\textrm{h}$, one side receives
$\lceil p_\textrm{h}/2\rceil$ rows and the other $\lfloor p_\textrm{h}/2\rfloor$,
and the output is shifted by half a pixel. This is why CNNs almost always use
odd kernels, of size 1, 3, 5, or 7 with padding 0, 1, 2, or 3 on each side.

To check the formula in code, we need a helper. Convolutional layers expect
a minibatch of examples with several channels, so `comp_conv2d` adds two
dimensions of size 1 to a two-dimensional input, applies the layer, and
removes them again.

In [2]:
def comp_conv2d(conv2d, X):
    """Apply a convolutional layer to a single-channel 2D input."""
    Y = conv2d(tf.reshape(X, (1,) + X.shape + (1,)))  # input (1, h, w, 1)
    return tf.reshape(Y, Y.shape[1:3])

An $8\times 8$ input, a $3\times 3$ kernel, and one pixel of padding on every
side ($p_\textrm{h}=p_\textrm{w}=2$ in total; padding arguments in code count
the rows or columns on each side, here 1) should give an $8\times 8$ output.

In [3]:
X = tf.random.uniform(shape=(8, 8))
conv2d = tf.keras.layers.Conv2D(1, kernel_size=3, padding='same')
comp_conv2d(conv2d, X).shape

TensorShape([8, 8])

Height and width are padded independently. A kernel of height 5 and width 3
keeps the $8\times 8$ shape with two rows of padding above and below and one
column on either side.

In [4]:
conv2d = tf.keras.layers.Conv2D(1, kernel_size=(5, 3), padding='same')
comp_conv2d(conv2d, X).shape

TensorShape([8, 8])

Keras `Conv2D` accepts only `padding='valid'`, which means no padding, and
`padding='same'`, which pads so that the output has $\lceil n/s\rceil$ rows
and columns for stride $s$. At stride 1 and odd kernel sizes this is the
symmetric padding $(k-1)/2$ used above. Other amounts of padding take a
separate `ZeroPadding2D` layer, which pads each side by a given amount,
followed by `padding='valid'`, as in the stride examples below. Keras layers
take channels-last inputs: (example, height, width, channel).

### Padding Values

Zero padding treats the image as if it were black outside its frame. This is
a *boundary condition*: a choice of values beyond the border, which the image
itself does not provide. To see its effect, we cross-correlate a uniformly
bright $5\times 5$ image of ones, padded by one pixel of zeros on every side,
with a $3\times 3$ kernel of ones, using `corr2d` from
that section. Each output then counts the image pixels in its
window.

In [5]:
img = d2l.ones((5, 5))
d2l.corr2d(tf.pad(img, [[1, 1], [1, 1]]), d2l.ones((3, 3)))

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[4., 6., 6., 6., 4.],
       [6., 9., 9., 9., 6.],
       [6., 9., 9., 9., 6.],
       [6., 9., 9., 9., 6.],
       [4., 6., 6., 6., 4.]], dtype=float32)>

The interior outputs are 9, those along the edges 6, and those in the corners
4: the layer sees a dark frame around an image that has none. The frame
travels inward through later layers, which can learn to detect it, and the
resulting artifacts near the border can leave parts of the input to which the
network barely responds [@Alsallakh.Kokhlikyan.Miglani.ea.2020]. Two
other boundary conditions are common. *Reflect padding* mirrors the rows and
columns next to the border, and *replicate padding* repeats the border pixels
themselves. Both keep a uniform image uniform, and image-generation networks
often use them because a dark frame would be visible in their output. Zero
padding remains the default in classifiers: it is cheap, and libraries apply
it without building a larger tensor.

## Stride

### Strided Convolution

A network that keeps a $224\times 224$ image at full resolution computes
50,176 outputs per channel in every layer. Since each $3\times 3$ layer widens
the region a unit sees by only two pixels, such a network would need more than
100 layers before a single unit could see the whole image, as a classifier
must. Moving the window in larger steps reduces both the number of outputs
and the number of layers. So far the window has moved one pixel at a time, to
the right and then down. The number of rows and columns it moves per step is the *stride*. With stride
$s_\textrm{h}$ along the height and $s_\textrm{w}$ along the width, the layer
evaluates only every $s_\textrm{h}$-th row and every $s_\textrm{w}$-th column
of window positions, so it computes about $s_\textrm{h} s_\textrm{w}$ times
fewer outputs. the figure uses stride 3 along the height and 2
along the width on the padded input of the figure. The two shaded
outputs are $0\times0+0\times1+1\times2+2\times3=8$, whose window lies two
columns to the right of the first window, and
$0\times0+6\times1+0\times2+0\times3=6$, whose window lies three rows below
it. Moving two more columns to the right would put the window partly
outside the padded input, so each row has two outputs.

![Cross-correlation with strides of 3 and 2 for height and width, respectively.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-stride.svg)

To count the outputs along the height, note that the window starts in row 0 of
the padded input and that its last row must stay inside the
$n_\textrm{h}+p_\textrm{h}$ rows. The possible starting rows are
$0, s_\textrm{h}, 2s_\textrm{h}, \ldots$, up to $n_\textrm{h}+p_\textrm{h}-k_\textrm{h}$,
so with the same count for the width the output shape is

$$\left(\left\lfloor\frac{n_\textrm{h}-k_\textrm{h}+p_\textrm{h}}{s_\textrm{h}}\right\rfloor+1\right)\times\left(\left\lfloor\frac{n_\textrm{w}-k_\textrm{w}+p_\textrm{w}}{s_\textrm{w}}\right\rfloor+1\right).$$

The fraction counts the steps the window can take and the $+1$ counts its
starting position. For the figure this gives
$\lfloor 3/3\rfloor+1=2$ rows and $\lfloor 3/2\rfloor+1=2$ columns, and at
stride 1 it is the padding formula. With shape-preserving padding,
$p_\textrm{h}=k_\textrm{h}-1$, the number of rows is
$\lfloor (n_\textrm{h}-1)/s_\textrm{h}\rfloor+1=\lceil n_\textrm{h}/s_\textrm{h}\rceil$:
stride $s$ divides the resolution by $s$, rounding up.

The most common strided layer has a $3\times 3$ kernel, padding 1 on each
side, and stride 2. It halves an $8\times 8$ input to $4\times 4$.

In [6]:
conv2d = tf.keras.Sequential([
    tf.keras.layers.ZeroPadding2D(padding=1),
    tf.keras.layers.Conv2D(1, kernel_size=3, padding='valid', strides=2)])
comp_conv2d(conv2d, X).shape

TensorShape([4, 4])

At stride 2 on an input of even size, `padding='same'` pads one row and
column at the bottom and right but none at the top and left, so output $i$
is centered on input pixel $2i+1$ rather than $2i$. `ZeroPadding2D(1)`
followed by `padding='valid'` pads both sides by 1.

The settings can also differ between the axes. A $3\times 5$ kernel with
padding $(0, 1)$ on each side and stride $(3, 4)$ gives
$\lfloor (8-3+0)/3\rfloor+1=2$ rows and $\lfloor (8-5+2)/4\rfloor+1=2$
columns.

In [7]:
conv2d = tf.keras.Sequential([
    tf.keras.layers.ZeroPadding2D(padding=(0, 1)),
    tf.keras.layers.Conv2D(1, kernel_size=(3, 5), padding='valid',
                           strides=(3, 4))])
comp_conv2d(conv2d, X).shape

TensorShape([2, 2])

When the stride equals the kernel size and there is no padding, the windows
tile the input without overlap. A $16\times 16$ kernel with stride 16 cuts a
$224\times 224$ image into a $14\times 14$ grid of patches and maps each patch
to one output per channel. This *patchify* layer is how vision transformers
turn an image into a sequence of patches, and some convolutional networks
begin the same way.

When height and width use the same setting, we speak of padding $p$ and
stride $s$.

`tf.keras.layers.Conv2D` defaults to `padding='valid'`, no padding, and
stride 1.

### Stride and the Receptive Field

The receptive field of a unit, as defined in that section, is
the set of input pixels that can affect its value. A $k\times k$ layer at
stride 1 widens it by $k-1$ pixels along each axis, so $L$ stacked
$3\times 3$ layers see a region of side $2L+1$, which grows only linearly
with depth. After a layer with stride 2, neighboring positions of its output
lie two input pixels apart, and a $3\times 3$ kernel applied to that output
reaches $2\times 2=4$ input pixels further instead of 2. In general, one step
in the input of layer $i$ spans $\prod_{j<i} s_j$ pixels of the original
input, and the top unit of a stack of $L$ layers with kernel sizes $k_i$ and
strides $s_i$ has a receptive field of side

$$r = 1 + \sum_{i=1}^{L} (k_i - 1) \prod_{j=1}^{i-1} s_j,$$

where the empty product for $i = 1$ equals 1. Each layer adds the extent of
its kernel, measured in pixels of the original input; at stride 1 every term
is $k_i-1$, as in that section. Padding does not appear, because it changes
which outputs exist, not how far each one sees. For three $3\times 3$ layers,
stride 1 everywhere gives $r = 1+2+2+2 = 7$, while strides 2, 2, 1 give
$r = 1+2+4+8 = 15$. The strided stack is drawn in panel (a) of
the figure below. Every stride multiplies the reach of all
later kernels, so a downsampling network's receptive field grows
geometrically with depth, which is how classifiers see the whole image after
a few dozen layers.

## Dilation

### Dilated Convolution

Stride lets a unit see far by giving up resolution. Some tasks need both: a
wide view of the image and an output at every pixel. A large kernel would
give the wide view, but its weights and its computation grow with the square
of its size. Padding and stride change where the kernel is applied. Dilation
changes the kernel's footprint itself: a *dilated convolution* [@yu2016dilated]
places the kernel's weights $d$ pixels apart instead of at adjacent pixels,
where $d$ is the *dilation*. A $k\times k$ kernel with dilation $d$ still
has $k^2$ weights, but along each axis they now span

$$k' = k + (k-1)(d-1) = d(k-1)+1$$

pixels, the *effective kernel size*. Along each axis, $d-1$ pixels are
skipped between neighboring weights, and $d=1$ is the ordinary convolution.
the figure shows a $3\times 3$ kernel at dilations 1 and 2.
At dilation 2 the nine weights spread over a $5\times 5$ footprint, skipping
every other pixel, while each output still takes nine multiplications.

![A $3\times 3$ kernel at dilation 1 and dilation 2. At dilation 2 its nine weights span a $5\times 5$ footprint.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-conv-dilation.svg)

For output shapes, a dilated kernel behaves like a dense kernel of its
effective size. Substituting $k'$ for $k$ in the strided formula gives the
output shape for padding, stride, and dilation together:

$$\left(\left\lfloor\frac{n_\textrm{h}+p_\textrm{h}-d_\textrm{h}(k_\textrm{h}-1)-1}{s_\textrm{h}}\right\rfloor+1\right)\times\left(\left\lfloor\frac{n_\textrm{w}+p_\textrm{w}-d_\textrm{w}(k_\textrm{w}-1)-1}{s_\textrm{w}}\right\rfloor+1\right).$$

With dilation 1 this is the strided formula, and with stride 1 and no padding
the $n-k+1$ rule we started from. On our $8\times 8$ input, a $3\times 3$
kernel at dilation 2 and a dense $5\times 5$ kernel should both give a
$4\times 4$ output.

In [8]:
conv2d = tf.keras.layers.Conv2D(1, kernel_size=3, dilation_rate=2)
conv5 = tf.keras.layers.Conv2D(1, kernel_size=5)
comp_conv2d(conv2d, X).shape, comp_conv2d(conv5, X).shape

(TensorShape([4, 4]), TensorShape([4, 4]))

### Dilation and the Receptive Field

A dilated layer at stride 1 widens the receptive field by $k'-1 = d(k-1)$
pixels, so the equation holds with each $k_i$ replaced by its
effective size. Doubling the dilation from layer to layer,
$d_i = 1, 2, 4, \ldots, 2^{L-1}$, gives $L$ layers of $3\times 3$ kernels a
receptive field of side
$1 + 2(1 + 2 + \cdots + 2^{L-1}) = 2^{L+1}-1$. It grows exponentially with
depth, like that of a strided stack, but every layer keeps the resolution of
the input.

the figure compares the two ways to reach far. With three
layers of size 3, strides 2, 2, 1 and dilations 1, 2, 4 both give a receptive
field of 15 pixels, and the top unit depends on the input through the same
connections. The strided stack keeps only every fourth position; the dilated
stack computes all of them. Without padding and with the same weights, the
strided stack returns exactly every fourth output of the dilated stack, even
with nonlinearities between the layers: dilation is a strided stack that
discards nothing. This matters for *dense prediction* tasks such as semantic
segmentation, which assigns a label to every pixel and thus needs both wide
context and full resolution, and dilated convolutions are widely used there
[@yu2016dilated]. Their cost is that every layer computes its outputs at
full resolution. A network that downsamples with strides can instead restore
the resolution at its end with learned upsampling, the transposed
convolution of that section.

![Receptive field of the top unit of three stacked layers with kernel size 3, in one dimension. Cells are the positions each layer computes, shaded cells those the top unit depends on, and lines the kernel's taps. (a) Strides 2, 2, 1: layer 1 keeps every second position, layers 2 and 3 every fourth. (b) Stride 1 with dilations 1, 2, 4: every layer keeps all positions. Both top units see the same 15 input pixels.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-rf-towers.svg)

When height and width use the same dilation, we speak of dilation $d$; by
default it is 1. We can now predict the output shape of a single-channel
layer under any padding, stride, and dilation, and how far its units see.
Real inputs, however, have several channels, and a layer should produce
several feature maps, which is the subject of that section.

## Summary

A convolution shrinks its input by $k-1$ pixels along each axis and uses
border pixels in fewer windows than interior ones. Padding by $k-1$ in total,
split evenly for odd kernels, keeps the shape and centers each output on its
input pixel; zero padding imposes a boundary condition that the network can
detect. Stride $s$ evaluates every $s$-th window, divides the resolution and
the cost by about $s$ along each axis, and makes the receptive field of later
layers grow geometrically. Dilation $d$ spreads a kernel's weights to an
effective size $d(k-1)+1$, which enlarges the receptive field at full
resolution and at constant cost per output. One formula,
the equation, gives the output shape for all three.

## Exercises

1. [code] **Mirror padding.** Mirror padding, also called reflect padding,
   extends a tensor by reflecting its entries about the border without
   repeating the border entry: padding the vector $(1, 2, 3)$ by one entry
   on each side gives $(2, 1, 2, 3, 2)$.
    1. Implement mirror padding by one entry on every side of a
       two-dimensional tensor, and check the result by hand on a
       $3 \times 3$ tensor with distinct entries.
    1. Repeat the computation of that section with
       mirror padding instead of zero padding, and explain the result.
    1. Take the $5\times 5$ ramp $x_{ij} = j$ and the $3\times 3$ kernel
       whose entries all equal $\frac{1}{9}$. Show that in the interior the
       output equals the input. Compute the outputs in the first and last
       columns under zero padding and under mirror padding, and find a way
       to pad that reproduces the ramp exactly.

1. **Benefits of stride.** A $3 \times 3$ convolution with one pixel of
   padding on every side is applied to a single-channel $64 \times 64$
   input.
    1. Count the multiplications this layer performs at stride 1 and at
       stride 2. By what factor does stride 2 reduce the cost of a second,
       stride-1 layer of the same kind placed after this one?
    1. Four such layers are stacked. Use the equation to compute
       the receptive field of one output element when all four layers have
       stride 1 and when the first two have stride 2. Use the comparison to
       give one statistical benefit of a stride larger than 1.

1. **Stride as sampling.** For a length-6 one-dimensional input, write the
   $4 \times 6$ weight matrix of a stride-1, kernel-3 convolution in the
   banded-matrix form of that section, and the
   $2 \times 4$ zero--one matrix that keeps every other entry of a
   length-4 vector. Show that composing the two reproduces the matrix of
   the stride-2, kernel-3 convolution.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 10.15.*

1. **Dilation and gridding.** A network stacks four $3 \times 3$
   convolutions with stride 1. The doubling dilation schedule
   $(1, 2, 4, 8)$ gives one output element a receptive field of side $31$.
   Compare it with the constant schedule $(4, 4, 4, 4)$.
    1. Use the equation, with each kernel replaced by its
       effective size, to compute the receptive field of the constant
       schedule.
    1. For each schedule, determine which pixels inside the receptive
       field the output actually depends on, and count them. A schedule
       whose output depends only on a sparse lattice within its receptive
       field is said to exhibit *gridding*.
    1. Give one reason gridding is harmful for dense prediction tasks such
       as segmentation.

1. **Strides versus dilations.** Consider the two one-dimensional stacks of
   the figure, without padding, with kernels
   $\mathbf{w}_1, \mathbf{w}_2, \mathbf{w}_3$ of size 3 shared by both
   stacks and an elementwise nonlinearity after every layer.
    1. Write the output of each stack at position $i$ as a function of the
       input, and show that the output of the strided stack at position $i$
       equals the output of the dilated stack at position $4i$.
    1. For an input of length $n$, count the multiplications each stack
       performs, ignoring the border.
    1. Generalize the first part: which dilations reproduce a stack with
       strides $s_1, \ldots, s_L$?

1. **Fractional stride.** ● Read a stride of $\frac{1}{2}$ as moving the
   convolution window by half an input element per output element.
    1. Evaluate the equation at $d_\textrm{h} = 1$,
       $s_\textrm{h} = \frac{1}{2}$, and $p_\textrm{h} = k_\textrm{h} - 1$.
       How does the output height compare with the input height?
    1. Describe how to compute this output with an ordinary stride-1
       convolution applied to a modified input, and check that your
       construction gives the same output height.
    1. Name a task in which a network needs this operation, and say where
       in the network it would be placed.

1. **Gridding-free dilations.** ● A network stacks $L$ convolutions of
   size $3 \times 3$ with stride 1 and dilations $d_1, \ldots, d_L$. State
   a condition on $d_1, \ldots, d_L$ that guarantees the output depends on
   every pixel of its receptive field, so that the stack does not exhibit
   gridding, and show that the condition suffices.

[Discussions](https://d2l.discourse.group/t/272)